# Aula 4 - Manipulação de df: groupby, merge e apply


### Objetivos

Apresentar como unir dataframes e realizar cálculos com dados agrupados.

____________________________

### Habilidades a serem desenvolvidas nessa aula

Ao final da aula o aluno deve:

- Saber como concatenar dataframes,
- Conseguir agrupar os dados e aplicar vários métodos à eles


____
____
____

Nas outras aulas vimos como extrair algumas informações utilizando linhas e colunas, mas e se quisessemos uma forma mais fácil de saber a média considerando um determinado grupo, por exemplo a média de sobreviventes por Pclass do Titanic?

## Como extrair informação dos dados?

In [1]:
# lê dataframe do arquivo titanic.csv 
import pandas as pd

df = pd.read_csv('data/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Como faríamos para calcular a média do preço pago para viajar no titanic para cada uma das classes utilizando apenas o que aprendemos até agora?

In [ ]:
df[df["Pclass"]==1][["Fare"]].mean()

Fare    84.154687
dtype: float64

In [6]:
df[df["Pclass"]==2][["Fare"]].mean()

Fare    20.662183
dtype: float64

In [7]:
df[df["Pclass"]==3][["Fare"]].mean()

Fare    13.67555
dtype: float64

Ou de forma mais automática:

In [ ]:
for i in [1, 2, 3]:
    valor_medio = df[df["Pclass"]==i]['Fare'].mean()
    print(f"A {i} classe pagou em média {valor_medio} dólares")

A 1 classe pagou em média 84.1546875 dólares
A 2 classe pagou em média 20.662183152173913 dólares
A 3 classe pagou em média 13.675550101832993 dólares


E se quisessemos calcular a média por Pclass e Sex?

### [Groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
Assim como no SQL, no pandas também temos um método com o qual podemos agregar os dados. O `groupby` primeiro separa nossos dados em grupos definidos dentro do método,  aplica uma operação usando agregação, transformação, filtragem ou até uma função própria e, por fim, junta os resultados encontrados.
<br>

<img src="groupby.png"  style="width: 700px" >

Exemplo de agregação da coluna de sexo e aplicação da função de agregação `mean` para obter a média das alturas.

<br>

Utilizar o `groupby` é o mesmo que fazer a sequência:

   1. Dividir os dados em grupos utilizando um critério
    
   2. Aplicar uma função em cada um dos grupos separadamente
    
   3. Combinar o resultado em uma estrutura de dados

<br>
Estrutura:

```python
df.groupby([lista_colunas_queremos_agregar]).funcao_agregacao()
```

Exemplo:

```python
df.groupby(['Gender']).mean()
```

#### Funções de agregação
Com essas funções podemos aplicar operações estatísticas nos nossos dados. Exemplos:

`mean`, `std`, `max`, `min`, `count`, `sum`, `var`.

Quando queremos aplicar apenas uma dessas operações podemos chamá-las diretamente após o `groupby`:


In [12]:
# Agrupa por Pclass e como feito no exemplo com o loop for
df.groupby('Pclass')[["Fare"]].mean()

,Fare
Pclass,
1,84.154687
2,20.662183
3,13.675550


In [14]:
df.groupby(['Pclass'])[["Fare"]].mean()

,Fare
Pclass,
1,84.154687
2,20.662183
3,13.675550


In [17]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [16]:
# Agrupa por Pclass e Sex e calcula a média de cada grupo
df.groupby(['Pclass', 'Sex'])[["Fare"]].mean()

Fare
Pclass Sex               
1      female  106.125798
       male     67.226127
2      female   21.970121
       male     19.741782
3      female   16.118810
       male     12.661633

In [19]:
df.groupby(['Pclass', 'Sex']).mean(numeric_only=True)

PassengerId  Survived        Age     SibSp     Parch  \
Pclass Sex                                                            
1      female   469.212766  0.968085  34.611765  0.553191  0.457447   
       male     455.729508  0.368852  41.281386  0.311475  0.278689   
2      female   443.105263  0.921053  28.722973  0.486842  0.605263   
       male     447.962963  0.157407  30.740707  0.342593  0.222222   
3      female   399.729167  0.500000  21.750000  0.895833  0.798611   
       male     455.515850  0.135447  26.507589  0.498559  0.224784   

                     Fare  
Pclass Sex                 
1      female  106.125798  
       male     67.226127  
2      female   21.970121  
       male     19.741782  
3      female   16.118810  
       male     12.661633

Aqui agregamos os dados por Pclass e Sex e em todas as colunas numéricas foi calculada a média. Se quiséssemos a média de apenas uma coluna poderíamos adicioná-la ao final da nossa sentença:

In [ ]:
# Queremos apenas a média de idade considerando a classe e o sexo
df.groupby(['Pclass', 'Sex']).mean(numeric_only=True)[["Age"]]

Age
Pclass Sex              
1      female  34.611765
       male    41.281386
2      female  28.722973
       male    30.740707
3      female  21.750000
       male    26.507589

Ou de modo mais eficiente:

In [ ]:
df.groupby(['Pclass', 'Sex'])[["Age"]].mean()

Age
Pclass Sex              
1      female  34.611765
       male    41.281386
2      female  28.722973
       male    30.740707
3      female  21.750000
       male    26.507589

In [26]:
df_agrupado_pclass_sex = df.groupby(['Pclass', 'Sex'])[["Fare", "Age"]].mean()
df_agrupado_pclass_sex

Fare        Age
Pclass Sex                          
1      female  106.125798  34.611765
       male     67.226127  41.281386
2      female   21.970121  28.722973
       male     19.741782  30.740707
3      female   16.118810  21.750000
       male     12.661633  26.507589

In [27]:
df_agrupado_pclass_sex.index

MultiIndex([(1, 'female'),
            (1,   'male'),
            (2, 'female'),
            (2,   'male'),
            (3, 'female'),
            (3,   'male')],
           names=['Pclass', 'Sex'])

In [28]:
df_agrupado_pclass_sex.columns

Index(['Fare', 'Age'], dtype='object')

In [25]:
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


Note que `df.groupby('A').colname.mean()` é mais eficiente que `df.groupby('A').mean().colname` pois a agregação só será realizada na coluna de interesse (colname).

Reparem que a coluna utilizada no `groupby` virou um index do nosso df. Para convertê-la em coluna novamente temos duas formas: <br>
  1. chamar o parâmetro `as_index=False` dentro do `groupby`
  2. aplicar `.reset_index()` ao final da sentença

In [31]:
df_agrupado_pclass_sex = df.groupby(['Pclass', 'Sex'], as_index=False)[["Fare", "Age"]].mean()
df_agrupado_pclass_sex

,Pclass,Sex,Fare,Age
0,1,female,106.125798,34.611765
1,1,male,67.226127,41.281386
2,2,female,21.970121,28.722973
3,2,male,19.741782,30.740707
4,3,female,16.118810,21.750000
5,3,male,12.661633,26.507589


In [32]:
df_agrupado_pclass_sex = df.groupby(['Pclass', 'Sex'])[["Fare", "Age"]].mean().reset_index()
df_agrupado_pclass_sex

,Pclass,Sex,Fare,Age
0,1,female,106.125798,34.611765
1,1,male,67.226127,41.281386
2,2,female,21.970121,28.722973
3,2,male,19.741782,30.740707
4,3,female,16.118810,21.750000
5,3,male,12.661633,26.507589


### Aprofundamento de agregações


Quando queremos aplicar mais de uma operação chamamos o método `.agg()`

In [ ]:
df.groupby(["Pclass"])[['Fare']].agg(['mean','max','min'])

Fare               
             mean       max  min
Pclass                          
1       84.154687  512.3292  0.0
2       20.662183   73.5000  0.0
3       13.675550   69.5500  0.0

In [34]:
df.groupby(["Pclass", "Sex"])[['Fare']].agg(['mean','max','min'])

Fare                   
                     mean       max      min
Pclass Sex                                  
1      female  106.125798  512.3292  25.9292
       male     67.226127  512.3292   0.0000
2      female   21.970121   65.0000  10.5000
       male     19.741782   73.5000   0.0000
3      female   16.118810   69.5500   6.7500
       male     12.661633   69.5500   0.0000

In [35]:
df[(df.Fare ==0) & (df.Pclass==1)]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
263,264,0,1,"Harrison, Mr. William",male,40.0,0,0,112059,0.0,B94,S
633,634,0,1,"Parr, Mr. William Henry Marsh",male,NaN,0,0,112052,0.0,NaN,S
806,807,0,1,"Andrews, Mr. Thomas Jr",male,39.0,0,0,112050,0.0,A36,S
815,816,0,1,"Fry, Mr. Richard",male,NaN,0,0,112058,0.0,B102,S
822,823,0,1,"Reuchlin, Jonkheer. John George",male,38.0,0,0,19972,0.0,NaN,S


- Harrison, Mr. William -> secretário do J. Bruce Ismay

- Fry, Mr. Richard	 -> mordomo do J. Bruce Ismay

- Parr, Mr. William Henry Marsh - > engenheiro elétrico do navio

- Andrews, Mr. Thomas Jr -> foi um construtor naval irlandês e diretor-gerente da Harland and Wolff, principal arquiteto do RMS Titanic. Ele viajou na viagem inaugural para observar o desempenho do navio e morreu no naufrágio em 15 de abril de 1912. Ele foi considerado um herói por ajudar passageiros durante o desastre.

- Reuchlin, Jonkheer. John George	 -> Reuchlin was a director of the Holland America Line (HAL) and sailed at the invitation of the White Star Line to study the Titanic's technical features for HAL's future ships.

Para operações distintas em colunas distintas passamos um dicionário com o nome da coluna como chave e a operação como valor

In [36]:
import numpy as np
df.groupby(['Pclass']).agg({
    'Embarked': pd.Series.mode,
    'Fare': np.mean,
    'Age': ['min','max'],
    'Survived': 'mean'
    })

/tmp/ipykernel_3466336/479275121.py:2: FutureWarning: The provided callable <function mean at 0x74a34afea560> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby(['Pclass']).agg({


Embarked       Fare   Age        Survived
           mode       mean   min   max      mean
Pclass                                          
1             S  84.154687  0.92  80.0  0.629630
2             S  20.662183  0.67  70.0  0.472826
3             S  13.675550  0.42  74.0  0.242363

In [37]:
import numpy as np
df.groupby(['Pclass']).agg({
    'Embarked': pd.Series.mode,
    'Fare': np.mean,
    'Age': ['min','max'],
    'Survived': 'mean'
    }).columns

/tmp/ipykernel_3466336/2687740289.py:2: FutureWarning: The provided callable <function mean at 0x74a34afea560> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby(['Pclass']).agg({


MultiIndex([('Embarked', 'mode'),
            (    'Fare', 'mean'),
            (     'Age',  'min'),
            (     'Age',  'max'),
            ('Survived', 'mean')],
           )

Se quisermos que o df de saída tenha nomes específicos devemos seguir o padrão:

In [38]:
df.groupby(['Pclass']).agg(
                            mode_embarked=('Embarked',pd.Series.mode), 
                            mean_fare=('Fare',np.mean)
                          )

/tmp/ipykernel_3466336/937547654.py:1: FutureWarning: The provided callable <function mean at 0x74a34afea560> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df.groupby(['Pclass']).agg(


,mode_embarked,mean_fare
Pclass,,
1,S,84.154687
2,S,20.662183
3,S,13.675550


_____________
_____________
**Exercício:** Investigue se o "poder aquisitivo" e a "faixa etária" influenciaram na chance de sobrevivência. Para isso, agrupe os dados pela coluna Survived e calcule a média das colunas Fare e Age. O que os números sugerem?

In [39]:
df.groupby(["Survived"])[['Fare', 'Age']].mean()

,Fare,Age
Survived,,
0,22.117887,30.626179
1,48.395408,28.343690


In [40]:
df.groupby(["Survived"])[['Fare','Age']].agg(['mean','max','min'])

Fare                       Age            
               mean       max  min       mean   max   min
Survived                                                 
0         22.117887  263.0000  0.0  30.626179  74.0  1.00
1         48.395408  512.3292  0.0  28.343690  80.0  0.42

In [ ]:
df.groupby(['Survived'])[['Fare','Age']].describe()

Fare                                                            \
          count       mean        std  min      25%   50%   75%       max   
Survived                                                                    
0         549.0  22.117887  31.388207  0.0   7.8542  10.5  26.0  263.0000   
1         342.0  48.395408  66.596998  0.0  12.4750  26.0  57.0  512.3292   

            Age                                                      
          count       mean        std   min   25%   50%   75%   max  
Survived                                                             
0         424.0  30.626179  14.172110  1.00  21.0  28.0  39.0  74.0  
1         290.0  28.343690  14.950952  0.42  19.0  28.0  36.0  80.0

In [42]:
df.groupby(['Survived', 'Pclass'])[['Age']].describe()

Age                                                        
                 count       mean        std    min    25%    50%    75%   max
Survived Pclass                                                               
0        1        64.0  43.695312  15.284243   2.00  31.00  45.25  55.25  71.0
         2        90.0  33.544444  12.151581  16.00  25.00  30.50  39.00  70.0
         3       270.0  26.555556  12.334882   1.00  19.00  25.00  33.00  74.0
1        1       122.0  35.368197  13.760017   0.92  24.25  35.00  45.00  80.0
         2        83.0  25.901566  14.837787   0.67  17.50  28.00  34.00  62.0
         3        85.0  20.646118  11.995047   0.42  14.00  22.00  29.00  63.0

______________
_____________

## Cruzamento e concatenação de bases

Também é possível fazer **cruzamento de bases** com o pandas. 

Pra quem conhece SQL: esses são os joins!

Pra quem conhece Excel: essa é uma forma de fazer o procv!

Vamos supor que temos as notas de duas provas dos alunos separas em sheets diferentes do excel e queremos juntar essa notas em um único df.

In [2]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
# ler os dados de diferentes sheets do mesmo excel "notas.xlsx"
df1 = pd.read_excel("notas.xlsx", sheet_name="notas1")
df2 = pd.read_excel("notas.xlsx", sheet_name="notas2")

In [45]:
df1

,RA,aluno,prova1
0,1,joão,10
1,4,leia,10
2,2,maria,9
3,3,han,8
4,5,luke,7
5,7,obi wan,10


In [46]:
df2

,RA,aluno,prova2
0,1,joão,10
1,4,leia,10
2,2,maria,9
3,3,han,8
4,5,luke,7
5,6,anakin,10


Repare que temos alunos distintos nos dois df

Diferentes tipos de join

<img src="join_exemplo2.png" text="https://towardsdatascience.com/python-pandas-dataframe-join-merge-and-concatenate-84985c29ef78"/>

O pandas possui dois métodos específicos para trabalharmos com o join de colunas entre df: `.merge()` e `.join()`. O `.merge()` fornece mais flexibilidade de trabalho e iremos utilizar e ele.

Enquanto o `pd.join()` é um atalho para combinar dados baseados em índices, o `pd.merge()` é a ferramenta completa que permite fazer vários tratamentos, como de colunas duplicadas através de suffixes.

### [pd.merge()](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)

```python
pd.merge(left, right, how="inner", on=None, left_on=None, right_on=None, suffixes=("_x", "_y"))
```

```sql
select *
from df1
full outer join df2 on=["RA", "aluno"]
```

In [ ]:
# Vamos utilizar um outer join para juntar as duas bases
df1.merge(df2, how='outer')

,RA,aluno,prova1,prova2
0,1,joão,10.0,10.0
1,2,maria,9.0,9.0
2,3,han,8.0,8.0
3,4,leia,10.0,10.0
4,5,luke,7.0,7.0
5,6,anakin,NaN,10.0
6,7,obi wan,10.0,NaN


In [ ]:
df1.merge(df2, how='outer', on=["RA", "aluno"])

,RA,aluno,prova1,prova2
0,1,joão,10.0,10.0
1,2,maria,9.0,9.0
2,3,han,8.0,8.0
3,4,leia,10.0,10.0
4,5,luke,7.0,7.0
5,6,anakin,NaN,10.0
6,7,obi wan,10.0,NaN


In [ ]:
df1.merge(df2, how='outer', on=["RA"])

,RA,aluno_x,prova1,aluno_y,prova2
0,1,joão,10.0,joão,10.0
1,2,maria,9.0,maria,9.0
2,3,han,8.0,han,8.0
3,4,leia,10.0,leia,10.0
4,5,luke,7.0,luke,7.0
5,6,NaN,NaN,anakin,10.0
6,7,obi wan,10.0,NaN,NaN


In [57]:
df1.merge(df2, how='outer', left_on=["RA"], right_on="RA")

,RA,aluno_x,prova1,aluno_y,prova2
0,1,joão,10.0,joão,10.0
1,2,maria,9.0,maria,9.0
2,3,han,8.0,han,8.0
3,4,leia,10.0,leia,10.0
4,5,luke,7.0,luke,7.0
5,6,NaN,NaN,anakin,10.0
6,7,obi wan,10.0,NaN,NaN


In [56]:
df1.merge(df2, how='outer', on=["RA"], suffixes=["_1", "_2"])

,RA,aluno_1,prova1,aluno_2,prova2
0,1,joão,10.0,joão,10.0
1,2,maria,9.0,maria,9.0
2,3,han,8.0,han,8.0
3,4,leia,10.0,leia,10.0
4,5,luke,7.0,luke,7.0
5,6,NaN,NaN,anakin,10.0
6,7,obi wan,10.0,NaN,NaN


Eu preciso saber todo mundo que fez a prova 1, mas não fez a prova 2. Como fazer isso?

In [59]:
df_merge = df1.merge(df2, how='outer', on=["RA", "aluno"])
df_merge

,RA,aluno,prova1,prova2
0,1,joão,10.0,10.0
1,2,maria,9.0,9.0
2,3,han,8.0,8.0
3,4,leia,10.0,10.0
4,5,luke,7.0,7.0
5,6,anakin,NaN,10.0
6,7,obi wan,10.0,NaN


In [61]:
df_merge[df_merge["prova2"].isna()]

,RA,aluno,prova1,prova2
6,7,obi wan,10.0,NaN


_________________________
_________________________
**Exercício:** Tente fazer um inner, left e right join entre os dois dataframes da aula e analise as diferenças nos resultados.


In [51]:
df1.merge(df2, how='inner')

,RA,aluno,prova1,prova2
0,1,joão,10,10
1,4,leia,10,10
2,2,maria,9,9
3,3,han,8,8
4,5,luke,7,7


In [ ]:
df1.merge(df2, how='left')

,RA,aluno,prova1,prova2
0,1,joão,10,10.0
1,4,leia,10,10.0
2,2,maria,9,9.0
3,3,han,8,8.0
4,5,luke,7,7.0
5,7,obi wan,10,NaN


In [53]:
df1.merge(df2, how='right')

,RA,aluno,prova1,prova2
0,1,joão,10.0,10
1,4,leia,10.0,10
2,2,maria,9.0,9
3,3,han,8.0,8
4,5,luke,7.0,7
5,6,anakin,NaN,10


_________________________
_________________________

## Aprofundamento

### [pd.concat()](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
Diferente do `.merge()` e `.join()` que operam apenas com colunas, com o `.concat()` podemos especificar se queremos concatenar em linhas ou colunas.
Na concatenação de colunas o `.concat()` somente considera o index dos df e, por isso, não podemos especificar colunas como feito com o `.merge()`.

> Equivale ao `UNION` do SQL quando o `axis=0`.

```python
pd.concat(
    objs,
    axis=0,
    join="outer",
    ignore_index=False,
    copy=True
)
```

In [ ]:
pd.concat([df1, df2], axis=1, join="inner")

Repare que ao concatenar diretamente pelo index ele juntou o aluno obi wan com o anakin. 

Ao concatenar dois df nas linhas, o `.concat()` irá considerar o nome das colunas. Se temos colunas com nomes distintos e utilizamos o parâmetro join='inner', ele irá ignorar essas colunas e trazer só o que é comum entre eles: 

In [ ]:
pd.concat([df1, df2], axis=0, join="inner")

Para que ele considere todas as colunas utilizamos o argumento 
```python 
join="outer" 
```

In [ ]:
pd.concat([df1, df2], join="outer")

## Aprofundamento

Suponha que você precisasse criar uma coluna com o último nome de cada pessoa. Como poderia fazer isso?

### Apply
O método `.apply()` recebe uma função como input e aplica ela para todo o df como se fosse um loop. Se você quiser que essa função seja aplicada ao longo das colunas deve considerar `axis=1` e ao longo das linhas `axis=0`

Uma grande funcionalidade do pandas é que com o método `apply()` podemos aplicar uma **função** (muitas vezes, uma **função lambda**) a uma coluna ou linha de um DataFrame



Vamos selecionar a coluna de idades e aplicar uma função lambda **a todos os elementos dessa coluna** e somar um valor a ela utilizando função lambda 
```python
lambda x: x + 2
```

In [ ]:
funcao_simples = lambda x: x + 2
funcao_simples(5)

In [ ]:
df["Age"].apply(lambda x: x + 2)

Essa função lambda é equivalente a aplicar uma função do tipo:

```python

def funcao(x):

    return x + 2
```

In [ ]:
def funcao(x):
    return x + 2

df.Age.apply(funcao)

Um outro exemplo:

Suponha que no seu banco de dados os dados sejam salvos todos com a letra minúscula. Como usar .`apply()` para padronizar a coluna `Name`?

In [ ]:
df['Nomes_padronizados'] = df['Name'].apply(lambda x: x.lower())
# df['Name'].str.lower() # seria a forma mais simples de fazer isso, mas o objetivo aqui é mostrar o uso do apply
df.head(2)

Agora vamos usar uma função lambda para **extrair o sobrenome** dos nomes dos passageiros

Pra extrair o sobrenome, note que este está separada do resto do nome por vírgula.

Para perceber isso, dê uma olhada na coluna de nomes:

In [ ]:
df["Name"]

Portanto, podemos usar a função para strings `split(",")`, com quebra na vírgula, e depois selecionar o primeiro elemento da lista gerada!

Vamos aproveitar e **criar uma nova coluna da base**, com os sobrenomes!

In [ ]:
df["Surname"] = df["Name"].apply(lambda x: x.split(",")[0])
df.head()

Eu posso criar uma nova coluna condicionalmente, por exemplo se eu quiser criar uma nova classe que vai categorizar quem pagou mais ou menos de 50 dólares na entrada.

In [ ]:
def grupo_fare(row):
    if row > 50:
        return 'Alto'
    else:
        return 'Baixo'

df['grupo_fare'] = df.Fare.apply(grupo_fare)
df.head(3)

________
________

**Exercício:** Converta os valores da coluna "Sex" de "male" para 1 e "female" para 0 utilizando o apply.

_________
_________

## Exercícios

1. Considere a existência de três tabelas distintas:
* customer.csv : Possui a informação dos clientes em duas colunas: customer id  customer name
* products.csv : Contém informação dos produtos vendidos pela empresa em três colunas - p_id (product id), product (name) e price
* sales.csv : Contém informações das vendas realizadas em seis colunas - sale_id, c_id (customer id), p_id (product_id), qty (quantity sold), store (name)

Tome algum tempo para usar os métodos apresentados até agora para entender cada uma das bases antes de começar.

Conhecendo as bases e utilizando os métodos de concatenação de bases responda:


a) Usando a tabela sales, calcule o total de itens vendidos para cada produto.

b) Descubra quantos itens no total cada loja (`store`) vendeu.

c) Quais produtos não foram vendidos? Para responder, junto os datasets 'sales' e 'products' para verificar quais produtos de 'products' não constam em 'sales'. Você pode utilizar o método `.isna()` para filtrar o dataframe.

In [ ]:
import pandas as pd 
sales = pd.read_csv("data/sales.csv") 
products = pd.read_csv("data/products.csv") 

d) Quantos clientes não realizaram uma compra? Utilize a mesma estratégia anterior com as tabelas `sales` e `customers`

e) Liste a quantidade vendida e o faturamento de cada produto 

f) Liste a quantidade vendida de cada produto por loja

g) Qual loja teve maior faturamento?

h) Qual produto foi o mais vendido?

## Referências
https://pandas.pydata.org/docs/user_guide/groupby.html <br>
https://pandas.pydata.org/docs/user_guide/merging.html <br> 
https://towardsdatascience.com/python-pandas-dataframe-join-merge-and-concatenate-84985c29ef78 <br>
[When to use pandas transform function](https://towardsdatascience.com/when-to-use-pandas-transform-function-df8861aa0dcf) <br>
[Compara a performance entre várias formas de iterar em um df. Vai desde o for até apply e transform](https://youtu.be/rsyvErL0Bo8) <br>

_________________________
_________________________

## [Avaliação anônima](https://forms.gle/tShxhxNYhvi6ZmQm8)

- olhar tema da proficiência
- colocar projeto

- se tem simulado
- proficiencia com tempo durante a aula ou pode fazer qq horario